# 03.03 -- GQLAlchemy Compatibility

**Purpose:** Demonstrate that Orthograph models are fully compatible with
GQLAlchemy without a database connection. This notebook covers:

- Auto-generating GQLAlchemy `Node`/`Relationship` classes from Orthograph models
- Instantiating generated classes (Pydantic v1/v2 coexistence)
- Pre-save data validation using the Orthograph schema
- Cypher generation and static Cypher query validation
- Static query validation via `ValidatedQueryBuilder`

**No database connection is required.** Everything runs locally.

For database interaction (save, load, query execution), see
[03.04 -- GQLAlchemy Database Interaction](03.04_gqlalchemy_database_interaction.ipynb).

```bash
pip install orthograph[gqlalchemy,cypher]
```

## 1. Define the Schema

The schema is defined once using Orthograph's `NodeModel` and `RelationshipModel`.
This is the **single source of truth** for both validation and OGM.

In [ ]:
from typing import Optional

from orthograph import (
    Cardinality,
    GraphDataModel,
    NodeModel,
    RelationshipModel,
)


class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int
    tagline: Optional[str] = None


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_type__ = Person
    __target_type__ = Movie
    __source_cardinality__ = Cardinality.ZERO_OR_MORE
    __target_cardinality__ = Cardinality.ONE_OR_MORE
    role: str


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_type__ = Person
    __target_type__ = Movie


model = GraphDataModel(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn, Directed],
)

print(f"Model: {model.name}")
print(f"Node types: {model.node_labels}")
print(f"Relationship types: {model.relationship_labels}")

## 2. Visualize the Schema

In [ ]:
from orthograph.visualization import render


print(render(model, format="text"))

## 3. Auto-Generate GQLAlchemy Classes

`generate_gqlalchemy_classes()` translates Orthograph Pydantic v2 models into
GQLAlchemy Pydantic v1 `Node`/`Relationship` subclasses at runtime.
Both Pydantic versions coexist in the same process without conflict.

These generated classes are used internally by `GqlAlchemyClient` -- users
never need to import or interact with them directly.

In [ ]:
from orthograph.extensions.gqlalchemy import generate_gqlalchemy_classes


schema = generate_gqlalchemy_classes(model)

print("Generated node classes:", list(schema.node_classes.keys()))
print("Generated rel classes: ", list(schema.rel_classes.keys()))

In [ ]:
PersonGqa = schema.get_node_class("Person")

print(f"Class name:   {PersonGqa.__name__}")
print(f"GQA label:    {PersonGqa.label}")
print(f"GQA labels:   {PersonGqa.labels}")
print(f"Annotations:  {PersonGqa.__annotations__}")

p = PersonGqa(name="Alice", born=1985)
print(f"\nInstance:     {p}")
print(f"Properties:   {p._properties}")

In [ ]:
ActedInGqa = schema.get_rel_class("ACTED_IN")

print(f"Class name:   {ActedInGqa.__name__}")
print(f"GQA type:     {ActedInGqa.type}")
print(f"Annotations:  {ActedInGqa.__annotations__}")

r = ActedInGqa(_start_node_id=0, _end_node_id=1, role="Neo")
print(f"\nInstance:     {r}")
print(f"Properties:   {r._properties}")

## 4. Pre-Save Validation

Orthograph's `GraphValidator` runs entirely in-process. Bad data is rejected
**before** any database call is attempted. This is the same validation that
`GqlAlchemyClient.save_node()` performs internally.

In [ ]:
from orthograph.core.validator import GraphValidator


validator = GraphValidator(model)

result = validator.validate_nodes(
    [{"__label__": "Person", "name": "Alice", "born": 1985}]
)
print(f"Valid data:   is_valid={result.is_valid}, errors={len(result.errors)}")

result = validator.validate_nodes([{"__label__": "Movie", "title": "Inception"}])
print(f"Missing year: is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

result = validator.validate_nodes(
    [{"__label__": "Movie", "title": "Inception", "year": "twenty"}]
)
print(f"Wrong type:   is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

result = validator.validate_nodes(
    [{"__label__": "Person", "name": "Alice", "born": 1985, "unknown": "x"}]
)
print(f"Extra props:  is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

result = validator.validate_nodes([{"__label__": "City", "name": "NYC"}])
print(f"Unknown type: is_valid={result.is_valid}, errors={len(result.errors)}")
for issue in result.errors:
    print(f"  -> {issue.code}: {issue.message}")

## 5. Cypher Generation

In [ ]:
from orthograph.extensions.cypher import CypherGenerator


gen = CypherGenerator(model)

print("Uniqueness constraints:")
for constraint in gen.generate_constraints():
    print(f"  {constraint}")

query, params = gen.merge_node({"__label__": "Person", "name": "Alice", "born": 1985})
print(f"\nMERGE query:  {query}")
print(f"MERGE params: {params}")

query = gen.match_node(Person)
print(f"\nMATCH query:  {query}")

## 6. Static Cypher Query Validation

In [ ]:
from orthograph.extensions.cypher import validate_cypher


result = validate_cypher(
    "MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p.name, m.title",
    model,
)
print(f"Valid query:   {len(result.errors)} errors")

result = validate_cypher(
    "MATCH (s:Studio)-[:PRODUCED]->(m:Movie) RETURN s.name",
    model,
)
print(f"Invalid query: {len(result.errors)} errors")
for issue in result.errors:
    print(f"  {issue.code}: {issue.message}")

## 7. ValidatedQueryBuilder -- Static Validation

`ValidatedQueryBuilder.validate_query()` validates Cypher against the schema
without executing anything. No database connection needed. Useful for CI/CD
checks or linting pipelines.

In [ ]:
from orthograph.extensions.gqlalchemy import ValidatedQueryBuilder


vqb = ValidatedQueryBuilder(model=model, db=None)

result = vqb.validate_query("MATCH (p:Person)-[:ACTED_IN]->(m:Movie) RETURN p, m")
print(f"Valid:   is_valid={result.is_valid}")

result = vqb.validate_query("MATCH (s:Studio) RETURN s")
print(f"Invalid: is_valid={result.is_valid}")
for issue in result.errors:
    print(f"  {issue.code}: {issue.message}")

result = vqb.validate_query("MATCH ()-[:PRODUCED]->() RETURN *")
print(f"Invalid: is_valid={result.is_valid}")
for issue in result.errors:
    print(f"  {issue.code}: {issue.message}")